In [1]:
%cd /home/parthgandhi/Projects/MLBot/mlbot/src

/home/parthgandhi/Projects/MLBot/mlbot/src


In [2]:
import polars as pl
import polars.selectors as cs
from ma_bands.features import add_indicators, detect_events

In [ ]:
data = pl.scan_parquet("test_data.parquet")

ma_window_size = 50
atr_window_size = 20
atr_multi = 0.5
use_ema = True

In [4]:
res = add_indicators(
    data=data,
    ma_window_size=ma_window_size,
    use_ema=use_ema,
    atr_window_size=atr_window_size,
    atr_multi=atr_multi,
)

res = detect_events(data=res)

In [14]:
SYMBOL_LIST = ["ITC", "SBIN"]

In [15]:
res.with_columns(pl.col("timestamp").dt.year().alias("year")).filter(
    pl.col("symbol").is_in(SYMBOL_LIST)
).group_by("year", "is_bounce").len().with_columns(
    (pl.col("len") * 100 / pl.col("len").sum()).over(partition_by="year").alias("pct"),
    pl.col("len").sum().over(partition_by="year").alias("total_count"),
).sort(["year", "is_bounce"], descending=[False, True]).collect()

year,is_bounce,len,pct,total_count
i32,bool,u32,f64,u32
2024,true,6,54.545455,11
2024,false,5,45.454545,11
2025,true,35,57.377049,61
2025,false,26,42.622951,61
2026,true,5,50.0,10
2026,false,5,50.0,10
